# BDC 2026 - Training Model Baru (ConvNeXtV2-Base, ConvNeXtV2-Large, SwinV2-Large)

Notebook ini **HANYA** melatih 3 model yang belum pernah kamu gunakan sebelumnya, lalu menyimpan
checkpoint + cache probabilitas (OOF & test) masing-masing. **Tidak melatih ulang** ConvNeXtV2-Tiny
atau SigLIP2 (yang sudah ada checkpoint-nya dari notebook stacking sebelumnya), dan **belum melakukan
stacking/blending** — itu langkah selanjutnya setelah ini selesai.

**Model yang dilatih di sini** (tag `timm` sudah dicek valid satu per satu ke Hugging Face):

| Model | Tag `timm` | Parameter | Resolusi |
|---|---|---|---|
| ConvNeXtV2-Base | `convnextv2_base.fcmae_ft_in22k_in1k` | ~88.7M | 224px |
| ConvNeXtV2-Large | `convnextv2_large.fcmae_ft_in22k_in1k` | ~197M | 224px |
| SwinV2-Large | `swinv2_large_window12to16_192to256.ms_in22k_ft_in1k` | ~195M | **256px** (wajib, arsitektur window-based) |

**Peringatan waktu & VRAM:** dua model "Large" ini jauh lebih berat dari ConvNeXtV2-Tiny/SigLIP2 yang
sudah kamu latih. Beberapa penyesuaian sudah dibuat supaya lebih realistis dijalankan di GPU tunggal:

- **Mixed precision (AMP)** diaktifkan (`CONFIG["use_amp"]=True`) — menghemat VRAM & mempercepat training.
- **Batch size lebih kecil** khusus untuk model Large (default 8, vs 16 untuk Base) — sesuaikan lagi turun
  kalau masih kena `CUDA out of memory`.
- **`use_kfold=False`** (1 split, full epoch) sebagai default, sama seperti keputusan kamu sebelumnya.
- Checkpoint & cache disimpan di folder terpisah (`checkpoints_large/`, `oof_probs_large/`,
  `test_probs_large/`) supaya tidak menambah kekacauan file di direktori utama.

**Saran:** jalankan `convnextv2_base` dulu sendirian (comment 2 entri lain di `CONFIG["model_configs"]`)
untuk pastikan semuanya lancar, baru commit ke 2 model Large yang jauh lebih lama.

In [1]:
import os

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
CONFIG = {
    "root": "BDC 2026",
    "num_workers": 0,  # wajib 0 di Windows + Jupyter/VS Code, lihat catatan notebook sebelumnya
    "n_splits": 5,
    "use_kfold": False,  # False = 1 split cepat, full epoch (menang di eksperimen sebelumnya)
    "seed": 42,
    "lr": 2e-5,
    "weight_decay": 1e-4,
    "use_amp": True,  # mixed precision -- WAJIB untuk model sebesar ini supaya muat VRAM & lebih cepat
    "norm_stats": {
        "imagenet": {"mean": (0.485, 0.456, 0.406), "std": (0.229, 0.224, 0.225)},
    },
    "model_configs": [
        {
            "name": "convnextv2_base.fcmae_ft_in22k_in1k",
            "norm": "imagenet",
            "img_size": 224,
            "batch_size": 16,
            "epochs_per_fold": 10,
        },
        {
            "name": "convnextv2_large.fcmae_ft_in22k_in1k",
            "norm": "imagenet",
            "img_size": 224,
            "batch_size": 8,  # ~197M parameter -> turunkan lagi (mis. 4) kalau CUDA OOM
            "epochs_per_fold": 8,
        },
        {
            "name": "swinv2_large_window12to16_192to256.ms_in22k_ft_in1k",
            "norm": "imagenet",
            "img_size": 256,  # WAJIB 256, jangan diubah -- lihat catatan arsitektur di atas
            "batch_size": 8,
            "epochs_per_fold": 8,
        },
    ],
    "checkpoint_dir": "checkpoints_large",
    "oof_probs_dir": "oof_probs_large",
    "test_probs_dir": "test_probs_large",
}

## Deteksi Kelas & DataFrame

Sama persis dengan notebook stacking sebelumnya — `get_class_order` baca prefix angka di nama folder
(`0_Recyclable`, dst.), dipakai konsisten di semua notebook project ini.

In [3]:
def get_class_order(config):
    train_dir = os.path.join(config["root"], "train")
    return sorted(os.listdir(train_dir))


def build_dataframe(train_dir, class_order):
    images, labels = [], []
    for c in class_order:
        folder = os.path.join(train_dir, c)
        for img in os.listdir(folder):
            images.append(os.path.join(folder, img))
            labels.append(c)
    df = pd.DataFrame({"image": images, "label": labels})

    label2id = {name: i for i, name in enumerate(class_order)}
    df["target"] = df["label"].map(label2id)
    return df


def build_test_dataframe(config):
    test_dir = os.path.join(config["root"], "test")
    test_images = sorted(
        os.listdir(test_dir),
        key=lambda x: int("".join(filter(str.isdigit, x)))
    )
    test_df = pd.DataFrame({"image": test_images})
    test_df["id"] = test_df["image"].apply(lambda x: int("".join(filter(str.isdigit, x))))
    return test_df

## Dataset

In [4]:
class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = cv2.imread(row["image"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, row["target"]


class TestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.loc[idx, "image"]
        image_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, image_name

## Transform

Ketiga model baru ini semuanya pretrained ImageNet biasa (bukan gaya SigLIP), jadi normalisasinya sama
(`norm_stats["imagenet"]`). Resolusi (`img_size`) diambil per model dari `CONFIG["model_configs"]`,
bukan satu ukuran global -- penting karena SwinV2-Large wajib 256px, beda dari 2 model ConvNeXt (224px).

In [5]:
def get_transforms(img_size, norm_type, config):
    stats = config["norm_stats"][norm_type]

    train_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=20, p=0.5),
        A.Normalize(mean=stats["mean"], std=stats["std"]),
        ToTensorV2(),
    ])

    valid_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=stats["mean"], std=stats["std"]),
        ToTensorV2(),
    ])

    return train_transform, valid_transform

## K-Fold Split

In [6]:
def get_skf(config):
    return StratifiedKFold(n_splits=config["n_splits"], shuffle=True, random_state=config["seed"])


def get_fold_splits(df, skf, use_kfold):
    all_splits = list(skf.split(df, df["target"]))
    if use_kfold:
        return all_splits
    return all_splits[:1]

## Device

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 3060


## Training & Validasi (dengan AMP)

Ditambah *mixed precision* (`torch.cuda.amp.autocast` + `GradScaler`) dibanding notebook stacking
sebelumnya -- perlu untuk model sebesar ini supaya lebih hemat VRAM & lebih cepat. Otomatis nonaktif
kalau tidak ada GPU (`use_amp` hanya berlaku efektif di CUDA).

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler, use_amp):
    model.train()
    running_loss = 0
    preds, labels = [], []

    progress = tqdm(loader, desc="Train")
    for images, target in progress:
        images = images.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, target)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pred = torch.argmax(outputs, dim=1)
        preds.extend(pred.cpu().numpy())
        labels.extend(target.cpu().numpy())
        progress.set_description(f"Loss {loss.item():.4f}")

    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(labels, preds, average="macro")
    return epoch_loss, epoch_f1

In [9]:
@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device, use_amp):
    model.eval()
    running_loss = 0
    preds, labels = [], []

    for images, target in loader:
        images = images.to(device)
        target = target.to(device)

        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, target)

        running_loss += loss.item()
        pred = torch.argmax(outputs, dim=1)
        preds.extend(pred.cpu().numpy())
        labels.extend(target.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(labels, preds, average="macro")
    return epoch_loss, epoch_f1

## Class Weight Helper

In [10]:
def make_class_weights(df_fold, n_classes, device):
    counts = df_fold["target"].value_counts().reindex(range(n_classes), fill_value=0).values.astype(float)
    counts = np.clip(counts, 1, None)
    weights = counts.sum() / (n_classes * counts)
    return torch.tensor(weights, dtype=torch.float32).to(device)

## Pipeline OOF K-Fold per Model

`model_cfg` sekarang berisi `img_size`/`batch_size`/`epochs_per_fold` PER MODEL (bukan satu nilai
global), karena ketiga model ini kebutuhan resolusi & VRAM-nya beda-beda jauh. Checkpoint & cache
probabilitas disimpan ke folder terpisah supaya tinggal di-load nanti pas mau digabung, tanpa perlu
training ulang.

In [11]:
def run_kfold_oof(model_cfg, df, test_df, test_image_dir, skf, device, config):
    model_name = model_cfg["name"]
    norm_type = model_cfg["norm"]
    img_size = model_cfg["img_size"]
    batch_size = model_cfg["batch_size"]
    epochs = model_cfg["epochs_per_fold"]

    train_transform, valid_transform = get_transforms(img_size, norm_type, config)

    n_train = len(df)
    n_classes = df["target"].nunique()
    n_test = len(test_df)

    splits = get_fold_splits(df, skf, config["use_kfold"])
    n_folds_used = len(splits)

    oof_probs = np.full((n_train, n_classes), np.nan)
    test_probs_per_fold = np.zeros((n_folds_used, n_test, n_classes))

    safe_name = model_name.replace("/", "_")
    os.makedirs(config["checkpoint_dir"], exist_ok=True)

    test_dataset = TestDataset(test_df, image_dir=test_image_dir, transform=valid_transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=config["num_workers"], pin_memory=True)

    for fold, (train_idx, valid_idx) in enumerate(splits):
        print(f"\n{'=' * 60}")
        print(f"{model_name} | Fold {fold + 1}/{n_folds_used}")
        print(f"{'=' * 60}")

        train_df_fold = df.iloc[train_idx].reset_index(drop=True)
        valid_df_fold = df.iloc[valid_idx].reset_index(drop=True)

        train_dataset = WasteDataset(train_df_fold, transform=train_transform)
        valid_dataset = WasteDataset(valid_df_fold, transform=valid_transform)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=config["num_workers"], pin_memory=True)
        valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=config["num_workers"], pin_memory=True)

        # img_size cuma di-pass eksplisit untuk SwinV2 (window-based, resolusi wajib
        # sesuai). ConvNeXt fully-convolutional -- resolusi cukup diatur lewat transform,
        # tidak perlu (dan sebagian versi timm tidak menerima) kwarg img_size di sini.
        if "swinv2" in model_name:
            model = timm.create_model(model_name, pretrained=True, num_classes=n_classes, img_size=img_size)
        else:
            model = timm.create_model(model_name, pretrained=True, num_classes=n_classes)
        model = model.to(device)

        weights = make_class_weights(train_df_fold, n_classes, device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        use_amp = config["use_amp"] and device.type == "cuda"
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

        best_f1 = 0
        ckpt_path = os.path.join(config["checkpoint_dir"], f"{safe_name}_fold{fold}.pth")

        for epoch in range(epochs):
            train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, use_amp)
            valid_loss, valid_f1 = valid_one_epoch(model, valid_loader, criterion, device, use_amp)
            scheduler.step()

            print(f"\nEpoch {epoch + 1}/{epochs}")
            print(f"Train Loss : {train_loss:.4f} | Train F1 : {train_f1:.4f}")
            print(f"Valid Loss : {valid_loss:.4f} | Valid F1 : {valid_f1:.4f}")

            if valid_f1 > best_f1:
                best_f1 = valid_f1
                torch.save(model.state_dict(), ckpt_path)
                print(f"Model terbaik disimpan (Valid F1: {best_f1:.4f})")

        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()

        fold_probs = []
        with torch.no_grad():
            for images, _ in valid_loader:
                images = images.to(device)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    probs = torch.softmax(model(images), dim=1)
                fold_probs.append(probs.float().cpu().numpy())
        oof_probs[valid_idx] = np.concatenate(fold_probs, axis=0)

        fold_test_probs = []
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    probs = torch.softmax(model(images), dim=1)
                fold_test_probs.append(probs.float().cpu().numpy())
        test_probs_per_fold[fold] = np.concatenate(fold_test_probs, axis=0)

        print(f"\nFold {fold + 1} selesai. Best Valid F1: {best_f1:.4f}")

        # bersihkan VRAM sebelum lanjut ke fold berikutnya -- penting untuk model sebesar ini
        del model
        torch.cuda.empty_cache()

    print(f"\n{'=' * 60}")
    print(f"{model_name} SELESAI")
    print(f"{'=' * 60}")

    valid_mask = ~np.isnan(oof_probs).any(axis=1)
    oof_preds = oof_probs[valid_mask].argmax(axis=1)
    oof_f1 = f1_score(df.loc[valid_mask, "target"], oof_preds, average="macro")
    print(f"OOF macro F1 ({valid_mask.sum()} baris tervalidasi): {oof_f1:.4f}")

    class_order = get_class_order(config)
    print(f"\n=== Classification Report OOF ({model_name}) ===")
    print(classification_report(df.loc[valid_mask, "target"], oof_preds, target_names=class_order))

    test_probs = test_probs_per_fold.mean(axis=0)

    # simpan cache OOF & test probs -- ini yang nanti dipakai buat digabung TANPA training ulang
    os.makedirs(config["oof_probs_dir"], exist_ok=True)
    os.makedirs(config["test_probs_dir"], exist_ok=True)
    np.save(os.path.join(config["oof_probs_dir"], f"{safe_name}_oof.npy"), oof_probs)
    np.save(os.path.join(config["test_probs_dir"], f"{safe_name}_test.npy"), test_probs)
    print(f"\nCache disimpan:")
    print(f"  {os.path.join(config['oof_probs_dir'], safe_name + '_oof.npy')}")
    print(f"  {os.path.join(config['test_probs_dir'], safe_name + '_test.npy')}")

    return oof_probs, test_probs, valid_mask

## Jalankan Training Semua Model

In [12]:
def run_all_models(config, df, test_df, skf, device):
    oof_probs_list = []
    test_probs_list = []
    valid_mask_list = []
    model_names = []

    test_dir = os.path.join(config["root"], "test")

    for model_cfg in config["model_configs"]:
        oof_probs, test_probs, valid_mask = run_kfold_oof(
            model_cfg=model_cfg,
            df=df,
            test_df=test_df,
            test_image_dir=test_dir,
            skf=skf,
            device=device,
            config=config,
        )
        oof_probs_list.append(oof_probs)
        test_probs_list.append(test_probs)
        valid_mask_list.append(valid_mask)
        model_names.append(model_cfg["name"])

    return oof_probs_list, test_probs_list, valid_mask_list, model_names

## Run

Melatih ketiga model berurutan. **Belum ada stacking/blending atau submission di sini** — cache
(checkpoint + OOF/test probs) disimpan ke disk, siap dipakai nanti begitu kamu minta digabungkan
dengan ConvNeXtV2-Tiny + SigLIP2 yang sudah ada.

In [13]:
class_order = get_class_order(CONFIG)
print("Label mapping:", {name: i for i, name in enumerate(class_order)})

train_dir = os.path.join(CONFIG["root"], "train")
df = build_dataframe(train_dir, class_order)
test_df = build_test_dataframe(CONFIG)

skf = get_skf(CONFIG)

oof_probs_list, test_probs_list, valid_mask_list, model_names = run_all_models(CONFIG, df, test_df, skf, device)

print("\n" + "=" * 60)
print("SEMUA MODEL SELESAI")
print("=" * 60)
for name in model_names:
    print(f"  - {name}")
print(f"\nCheckpoint  : {CONFIG['checkpoint_dir']}/")
print(f"OOF cache   : {CONFIG['oof_probs_dir']}/")
print(f"Test cache  : {CONFIG['test_probs_dir']}/")
print("\nBelum ada submission dibuat -- minta saya buatkan kode penggabungan (stacking/blending)")
print("dengan ConvNeXtV2-Tiny + SigLIP2 yang sudah ada kapan saja kamu siap.")

Label mapping: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}

convnextv2_base.fcmae_ft_in22k_in1k | Fold 1/1


C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Loss 0.0078: 100%|██████████| 1325/1325 [09:13<00:00,  2.39it/s]



Epoch 1/10
Train Loss : 0.1286 | Train F1 : 0.9548
Valid Loss : 0.0611 | Valid F1 : 0.9796
Model terbaik disimpan (Valid F1: 0.9796)


Loss 0.0002: 100%|██████████| 1325/1325 [07:16<00:00,  3.04it/s]



Epoch 2/10
Train Loss : 0.0434 | Train F1 : 0.9846
Valid Loss : 0.0577 | Valid F1 : 0.9802
Model terbaik disimpan (Valid F1: 0.9802)


...


Epoch 6/8
Train Loss : 0.0070 | Train F1 : 0.9974
Valid Loss : 0.0726 | Valid F1 : 0.9847


Loss 0.0000: 100%|██████████| 2649/2649 [19:23<00:00,  2.28it/s]



Epoch 7/8
Train Loss : 0.0039 | Train F1 : 0.9986
Valid Loss : 0.0766 | Valid F1 : 0.9856
Model terbaik disimpan (Valid F1: 0.9856)


Loss 0.0000: 100%|██████████| 2649/2649 [19:23<00:00,  2.28it/s]



Epoch 8/8
Train Loss : 0.0025 | Train F1 : 0.9993
Valid Loss : 0.0772 | Valid F1 : 0.9856

Fold 1 selesai. Best Valid F1: 0.9856

swinv2_large_window12to16_192to256.ms_in22k_ft_in1k SELESAI
OOF macro F1 (5298 baris tervalidasi): 0.9856

=== Classification Report OOF (swinv2_large_window12to16_192to256.ms_in22k_ft_in1k) ===
              precision    recall  f1-score   support

0_Recyclable       0.98      0.98      0.98      2006
1_Electronic       0.99      1.00      0.99       790
   2_Organic       0.98      0.98      0.98      2502

    accuracy                           0.98      5298
   macro avg       0.99      0.99      0.99      5298
weighted avg       0.98      0.98      0.98      5298


Cache disimpan:
  oof_probs_large\swinv2_large_window12to16_192to256.ms_in22k_ft_in1k_oof.npy
  test_probs_large\swinv2_large_window12to16_192to256.ms_in22k_ft_in1k_test.npy

SEMUA MODEL SELESAI
  - convnextv2_base.fcmae_ft_in22k_in1k
  - convnextv2_large.fcmae_ft_in22k_in1k
  - swinv2_large